# Discovery + Silver: `university.grades`

Ultima tabla del dominio `university`. Grano: una calificacion individual por inscripcion (`enrollment_id`), varias por assessment (quiz/parcial/final/tarea/proyecto).

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.university__grades", engine)
df.shape

(60000, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

grade_id                 object
assessment               object
score                    object
weight                   object
graded_at                object
enrollment_id            object
_source_file             object
_ingested_at     datetime64[ns]
_dag_run_id              object
dtype: object


,grade_id,assessment,score,weight,graded_at,enrollment_id,_source_file,_ingested_at,_dag_run_id
0,GRD-00000001,project,96.92,0.48,2024-10-15,ENR-00002100,university/grades.csv,2026-07-21 09:50:02.698455,manual__2026-07-21T09:49:58+00:00
1,GRD-00000002,project,71.84,0.26,2025-05-20,ENR-00005987,university/grades.csv,2026-07-21 09:50:02.698455,manual__2026-07-21T09:49:58+00:00
2,GRD-00000003,homework,76.6,0.27,2025-01-09,ENR-00017245,university/grades.csv,2026-07-21 09:50:02.698455,manual__2026-07-21T09:49:58+00:00
3,GRD-00000004,quiz,73.4,0.47,2024-12-27,ENR-00009730,university/grades.csv,2026-07-21 09:50:02.698455,manual__2026-07-21T09:49:58+00:00
4,GRD-00000005,homework,70.99,0.39,2024-07-11,ENR-00019216,university/grades.csv,2026-07-21 09:50:02.698455,manual__2026-07-21T09:49:58+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("grade_id duplicados:", df["grade_id"].duplicated().sum())

enrollments = pd.read_sql("SELECT enrollment_id FROM silver.university__enrollments", engine)
print("enrollment_id huerfanos:", (~df["enrollment_id"].isin(enrollments["enrollment_id"])).sum())

Nulos por columna:
grade_id         0
assessment       0
score            0
weight           0
graded_at        0
enrollment_id    0
_source_file     0
_ingested_at     0
_dag_run_id      0
dtype: int64

grade_id duplicados: 0
enrollment_id huerfanos: 0


## 3. `assessment`, `score`, `weight`: valores y rangos

In [4]:
print("Valores distintos de assessment:")
print(df["assessment"].value_counts())
print()

score = pd.to_numeric(df["score"], errors="coerce")
weight = pd.to_numeric(df["weight"], errors="coerce")
print("score fuera de [0, 100]:", ((score < 0) | (score > 100)).sum())
print("weight fuera de [0, 1]:", ((weight < 0) | (weight > 1)).sum())
print()
print(score.describe())

Valores distintos de assessment:
assessment
quiz        12132
project     12003
midterm     11970
homework    11966
final       11929
Name: count, dtype: int64



score fuera de [0, 100]: 0
weight fuera de [0, 1]: 0

count    60000.000000
mean        74.884157
std         11.818063
min         24.530000
25%         66.840000
50%         74.980000
75%         83.100000
max        100.000000
Name: score, dtype: float64


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas, `score`/`weight` dentro de rango, `assessment` con 5 valores consistentes). Solo tipado: `score`/`weight` a numerico, `graded_at` a fecha.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["grade_id", "enrollment_id", "assessment", "score", "weight", "graded_at"]].copy()

df_silver["assessment"] = df_silver["assessment"].str.strip().str.lower()
df_silver["score"] = pd.to_numeric(df_silver["score"], errors="raise")
df_silver["weight"] = pd.to_numeric(df_silver["weight"], errors="raise")
df_silver["graded_at"] = pd.to_datetime(df_silver["graded_at"]).dt.date

df_silver.head()

,grade_id,enrollment_id,assessment,score,weight,graded_at
0,GRD-00000001,ENR-00002100,project,96.92,0.48,2024-10-15
1,GRD-00000002,ENR-00005987,project,71.84,0.26,2025-05-20
2,GRD-00000003,ENR-00017245,homework,76.60,0.27,2025-01-09
3,GRD-00000004,ENR-00009730,quiz,73.40,0.47,2024-12-27
4,GRD-00000005,ENR-00019216,homework,70.99,0.39,2024-07-11


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver.isna().sum().sum() == 0
assert df_silver["grade_id"].is_unique
assert df_silver["enrollment_id"].isin(enrollments["enrollment_id"]).all()
assert df_silver["score"].between(0, 100).all()
assert df_silver["weight"].between(0, 1).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 60000 filas listas para silver


## 7. Escribir en `silver.university__grades`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "university.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.university__grades CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "university__grades",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
print("Escrito en silver.university__grades")

OK: university.sql ejecutado


Escrito en silver.university__grades


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.university__grades LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT grade_id) AS ids_unicos FROM silver.university__grades", engine))
check

   filas  ids_unicos
0  60000       60000


,grade_id,enrollment_id,assessment,score,weight,graded_at,_silver_loaded_at
0,GRD-00000001,ENR-00002100,project,96.92,0.48,2024-10-15,2026-07-21 09:51:18.064215+00:00
1,GRD-00000002,ENR-00005987,project,71.84,0.26,2025-05-20,2026-07-21 09:51:18.064215+00:00
2,GRD-00000003,ENR-00017245,homework,76.60,0.27,2025-01-09,2026-07-21 09:51:18.064215+00:00
3,GRD-00000004,ENR-00009730,quiz,73.40,0.47,2024-12-27,2026-07-21 09:51:18.064215+00:00
4,GRD-00000005,ENR-00019216,homework,70.99,0.39,2024-07-11,2026-07-21 09:51:18.064215+00:00
